## 1.1. Piezoconductivity Equation (Dirichlet)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \sin(x)$, boundary conditions $p(0,t) = 0$, $p(\pi,t) = 0$.

In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Dirichlet)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  inputs_left = tf.concat([x_left, t], axis=1)
  inputs_right = tf.concat([x_right, t], axis=1)
  p_left = model(inputs_left)
  p_right = model(inputs_right)

  return tf.reduce_mean(
      tf.square(p_left)
  ) + tf.reduce_mean(
      tf.square(p_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.sin(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    print(f"Epoch {epoch}, Loss: {loss.numpy()}")

Epoch 0, Loss: 0.7019239664077759
Epoch 50, Loss: 0.1945224106311798
Epoch 100, Loss: 0.12152188271284103
Epoch 150, Loss: 0.07733044028282166
Epoch 200, Loss: 0.06269638985395432
Epoch 250, Loss: 0.04293101280927658
Epoch 300, Loss: 0.036104246973991394
Epoch 350, Loss: 0.030284198001027107
Epoch 400, Loss: 0.019263453781604767
Epoch 450, Loss: 0.01753387413918972
Epoch 500, Loss: 0.014058955013751984
Epoch 550, Loss: 0.013581736944615841
Epoch 600, Loss: 0.009791533462703228
Epoch 650, Loss: 0.007584075443446636
Epoch 700, Loss: 0.006883909925818443
Epoch 750, Loss: 0.006665064953267574
Epoch 800, Loss: 0.004715892486274242
Epoch 850, Loss: 0.004498162306845188
Epoch 900, Loss: 0.0037408568896353245
Epoch 950, Loss: 0.003656272077932954


In [2]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.sin(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.2. Piezoconductivity Equation (Neumann)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos(x)$, boundary conditions $p_x(0,t) = 0$, $p_x(\pi,t) = 0$.

In [3]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Neumann)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  return tf.reduce_mean(
      tf.square(p_x_left)
  ) + tf.reduce_mean(
      tf.square(p_x_right)
  )

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    print(f"Epoch {epoch}, Loss: {loss.numpy()}")

Epoch 0, Loss: 2.058985710144043
Epoch 50, Loss: 0.41797932982444763
Epoch 100, Loss: 0.3053441345691681
Epoch 150, Loss: 0.17258378863334656
Epoch 200, Loss: 0.13721612095832825
Epoch 250, Loss: 0.10585635155439377
Epoch 300, Loss: 0.09554996341466904
Epoch 350, Loss: 0.07432559877634048
Epoch 400, Loss: 0.070649653673172
Epoch 450, Loss: 0.06817304342985153
Epoch 500, Loss: 0.05783984810113907
Epoch 550, Loss: 0.060748886317014694
Epoch 600, Loss: 0.04766133800148964
Epoch 650, Loss: 0.04157804325222969
Epoch 700, Loss: 0.038473471999168396
Epoch 750, Loss: 0.03161317855119705
Epoch 800, Loss: 0.025074515491724014
Epoch 850, Loss: 0.024502120912075043
Epoch 900, Loss: 0.022585036233067513
Epoch 950, Loss: 0.019645588472485542


In [4]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()

## 1.3. Piezoconductivity Equation (Robin)

Solve the 1D piezoconductivity equation: $p_t = \chi p_{xx}$ for $x$ in $[0, \pi]$, $t$ in $[0, 1]$, with initial condition $p(x, 0) = \cos\left(x - \frac{\pi}{4}\right)$, boundary conditions $p_x(0,t) - p(0,t) = 0$, $p_x(\pi,t) - p(\pi,t) = 0$.

In [5]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Parameters
chi = 1
domain_x = [0, np.pi]
domain_t = [0, 1]

# Neural Network
class PINN(tf.keras.Model):
  def __init__(self):
    super(PINN, self).__init__()
    self.dense1 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense2 = tf.keras.layers.Dense(20, activation='tanh')
    self.dense3 = tf.keras.layers.Dense(1)

  def call(self, inputs):
    x = self.dense1(inputs)
    x = self.dense2(x)
    return self.dense3(x)

# PDE residual
def pde_residual(model, x, t):
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x)
    tape.watch(t)
    inputs = tf.concat([x, t], axis=1)
    p = model(inputs)
    p_t = tape.gradient(p, t)
    p_x = tape.gradient(p, x)
  p_xx = tape.gradient(p_x, x)
  del tape
  return p_t - chi * p_xx

# Boundary condition loss (Robin)
def bc_loss(model):
  t = tf.random.uniform(
    (50, 1),
    domain_t[0],
    domain_t[1]
  )
  x_left = tf.zeros_like(t)
  x_right = tf.ones_like(t) * np.pi
  with tf.GradientTape(persistent=True) as tape:
    tape.watch(x_left)
    tape.watch(x_right)
    inputs_left = tf.concat([x_left, t], axis=1)
    inputs_right = tf.concat([x_right, t], axis=1)
    p_left = model(inputs_left)
    p_right = model(inputs_right)

    p_x_left = tape.gradient(p_left, x_left)
    p_x_right = tape.gradient(p_right, x_right)
  del tape

  loss_left = tf.reduce_mean(tf.square(p_x_left - p_left))
  loss_right = tf.reduce_mean(tf.square(p_x_right - p_right))

  return loss_left + loss_right

# Initial condition loss
def ic_loss(model):
  x = tf.random.uniform(
      (50, 1),
      domain_x[0],
      domain_x[1]
  )
  t = tf.zeros_like(x)
  inputs = tf.concat([x, t], axis=1)
  p = model(inputs)
  p_ic = tf.cos(x - np.pi / 4)
  return tf.reduce_mean(tf.square(p-p_ic))

# Training
model = PINN()
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
for epoch in range(1000):
  x = tf.random.uniform((100, 1), domain_x[0], domain_x[1])
  t = tf.random.uniform((100, 1), domain_t[0], domain_t[1])
  with tf.GradientTape() as tape:
    residual = pde_residual(model, x, t)
    loss_pde = tf.reduce_mean(tf.square(residual))
    loss_bc = bc_loss(model)
    loss_ic = ic_loss(model)
    loss = loss_pde + loss_bc + loss_ic
  grads = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  if epoch % 50 == 0:
    print(f"Epoch {epoch}, Loss: {loss.numpy()}")

Epoch 0, Loss: 8.845766067504883
Epoch 50, Loss: 0.506769061088562
Epoch 100, Loss: 0.35633260011672974
Epoch 150, Loss: 0.301363080739975
Epoch 200, Loss: 0.235205739736557
Epoch 250, Loss: 0.2571927309036255
Epoch 300, Loss: 0.2287544310092926
Epoch 350, Loss: 0.17474356293678284
Epoch 400, Loss: 0.15258094668388367
Epoch 450, Loss: 0.12296608090400696
Epoch 500, Loss: 0.132870152592659
Epoch 550, Loss: 0.11844242364168167
Epoch 600, Loss: 0.0948200523853302
Epoch 650, Loss: 0.08283056318759918
Epoch 700, Loss: 0.07218659669160843
Epoch 750, Loss: 0.06837670505046844
Epoch 800, Loss: 0.06300529837608337
Epoch 850, Loss: 0.045772358775138855
Epoch 900, Loss: 0.03632108122110367
Epoch 950, Loss: 0.031297389417886734


In [6]:
import plotly.express as px
import pandas as pd

time_steps = np.linspace(domain_t[0], domain_t[1], 51)
plot_data = []

for t_val in time_steps:
  x_plot = np.linspace(domain_x[0], domain_x[1], 100)[:, None]
  t_plot = t_val * np.ones_like(x_plot)
  inputs_plot = np.concatenate([x_plot, t_plot], axis=1)
  p_plot = model(inputs_plot).numpy()
  p_exact = np.exp(-t_val) * np.cos(x_plot - np.pi / 4)
  df_temp = pd.DataFrame({
      'x': x_plot.flatten(),
      'PINN': p_plot.flatten(),
      'Exact': p_exact.flatten(),
      'Time': t_val
  })
  plot_data.append(df_temp)

df_plot = pd.concat(plot_data)
df_melted = df_plot.melt(id_vars=['x', 'Time'], var_name='Solution Type', value_name='p')
fig = px.line(df_melted, x='x', y='p', animation_frame='Time', color='Solution Type',
              title='1D Piezoconductivity Equation Solution Over Time')
fig.update_layout(
    xaxis_title='x',
    yaxis_title='p(x, t)'
)
fig.show()